<a href="https://colab.research.google.com/github/EgorMehed/ML-homework/blob/main/%D0%9C%D0%B5%D1%85%D0%B5%D0%B4_%D0%84%D0%B3%D0%BE%D1%80_%D0%A4%D0%86%D0%A23_12_%D0%A2%D1%80%D0%B5%D0%BD%D1%83%D0%B2%D0%B0%D0%BB%D1%8C%D0%BD%D0%B8%D0%B9_%D0%BD%D0%BE%D1%83%D1%82%D0%B1%D1%83%D0%BA_%D0%9B%D0%B5%D0%BA%D1%86%D1%96%D1%8F_3_Pandas_EDA_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Тренувальний ноутбук до лекції 3
## Аналіз даних у pandas та EDA

**Дисципліна:** Машинне навчання  
**Редакція:** 2026  

Мета ноутбука - відпрацювати основні операції pandas на єдиному навчальному наборі даних: завантаження, первинне обстеження, індексацію, очищення, групування, об’єднання, переформатування та базову візуалізацію.

> Матеріал орієнтований на pandas 3.0+. Якщо у Google Colab встановлена інша версія, більшість прикладів все одно виконуватиметься, але dtype текстових стовпців може відрізнятися.

## 0. Підготовка середовища

Спочатку імпортуємо бібліотеки та перевіримо версію pandas.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("pandas:", pd.__version__)
print("numpy:", np.__version__)

## 1. Створення навчального набору даних

Щоб ноутбук був автономним і запускався у Colab без зовнішніх файлів, згенеруємо невеликий набір даних про клієнтів інтернет-магазину. У ньому навмисно є пропуски, дублікати та неакуратні текстові значення.

In [ ]:
rng = np.random.default_rng(42)

n = 120
cities = np.array(["Kyiv", "Lviv", "Odesa", "Dnipro", "Kharkiv"])
segments = np.array(["basic", "standard", "premium"])

customers = pd.DataFrame({
    "customer_id": np.arange(1001, 1001 + n),
    "age": rng.integers(18, 70, size=n),
    "city": rng.choice(cities, size=n, p=[0.30, 0.20, 0.18, 0.16, 0.16]),
    "segment": rng.choice(segments, size=n, p=[0.45, 0.40, 0.15]),
    "signup_date": pd.Timestamp("2022-01-01") + pd.to_timedelta(
        rng.integers(0, 1600, size=n), unit="D"
    ),
    "orders": rng.poisson(7, size=n),
    "spend": np.round(rng.gamma(shape=2.3, scale=280, size=n), 2),
    "churn": rng.choice([0, 1], size=n, p=[0.78, 0.22]),
})

# Навмисно додаємо проблеми якості даних
customers.loc[[5, 18, 44], "age"] = np.nan
customers.loc[[9, 27, 78, 101], "spend"] = np.nan
customers.loc[12, "city"] = " kyiv "
customers.loc[31, "city"] = "LVIV"
customers.loc[66, "segment"] = None
customers.loc[88, "spend"] = -150.0

# Додаємо два дублікати рядків
customers = pd.concat([customers, customers.iloc[[3, 15]]], ignore_index=True)

customers.to_csv("customers_eda.csv", index=False)
print("Файл створено: customers_eda.csv")
print("Розмір:", customers.shape)
customers.head()

## 2. Завантаження даних та первинна перевірка

Завантажимо CSV так, як це робиться у реальному проєкті.

In [ ]:
df = pd.read_csv(
    "customers_eda.csv",
    parse_dates=["signup_date"],
)

df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.describe()

### Зверніть увагу
- `shape` показує кількість об'єктів та ознак;
- `info()` допомагає побачити типи й пропуски;
- `describe()` дає базову статистику числових ознак;
- у pandas 3.0 текстові стовпці за замовчуванням можуть мати `str` dtype замість старого `object`.

## 3. Series, DataFrame, loc та iloc

In [ ]:
# Один стовпець -> Series
spend = df["spend"]
print(type(spend))

# Кілька стовпців -> DataFrame
small = df[["customer_id", "city", "spend"]]
print(type(small))
small.head()

In [ ]:
# loc: вибір за мітками / умовами
high_spend = df.loc[df["spend"] > 1000, ["customer_id", "city", "spend"]]
high_spend.head()

In [ ]:
# iloc: вибір за позиціями
df.iloc[:5, :4]

## 4. Фільтрація даних

In [ ]:
mask = (
    df["age"].between(25, 45)
    & df["city"].isin(["Kyiv", "Lviv"])
    & df["spend"].notna()
)

selected = df.loc[mask]
selected.head()

In [ ]:
# query часто зручний для читабельних числових умов
query_result = df.query("orders >= 8 and spend > 700")
query_result.head()

## 5. Пропуски, дублікати та базове очищення

In [ ]:
missing = pd.DataFrame({
    "count": df.isna().sum(),
    "share": df.isna().mean(),
}).sort_values("share", ascending=False)

missing

In [ ]:
print("Повних дублікатів:", df.duplicated().sum())
print("Дублікатів customer_id:", df.duplicated(subset=["customer_id"]).sum())

In [ ]:
clean = df.drop_duplicates().copy()

# Негативні витрати трактуємо як помилкове значення
clean.loc[clean["spend"] < 0, "spend"] = np.nan

# Очищення тексту
clean["city"] = clean["city"].str.strip().str.title()
clean["segment"] = clean["segment"].fillna("unknown")

print(clean.shape)
print(clean["city"].value_counts(dropna=False))
print(clean["segment"].value_counts(dropna=False))

### Примітка про ML
У цьому навчальному EDA можна показати заповнення пропусків. Але якщо значення для заповнення (медіана, середнє тощо) використовуватиметься моделлю, його слід оцінювати **лише на train**, бажано всередині `Pipeline`.

In [ ]:
# Для дослідницького аналізу створимо окремий стовпець
median_age = clean["age"].median()
clean["age_filled"] = clean["age"].fillna(median_age)

print("Медіана age:", median_age)
clean[["age", "age_filled"]].head(10)

## 6. Робота з датами та створення ознак

In [ ]:
reference_date = pd.Timestamp("2026-09-07")

clean["signup_year"] = clean["signup_date"].dt.year
clean["signup_month"] = clean["signup_date"].dt.month
clean["tenure_days"] = (reference_date - clean["signup_date"]).dt.days
clean["spend_per_order"] = clean["spend"] / clean["orders"].replace(0, np.nan)

clean[[
    "signup_date", "signup_year", "signup_month",
    "tenure_days", "spend_per_order"
]].head()

## 7. Сортування, groupby та агрегування

In [ ]:
top_customers = clean.sort_values("spend", ascending=False).head(10)
top_customers[["customer_id", "city", "segment", "spend"]]

In [ ]:
city_summary = (
    clean.groupby("city", dropna=False)
         .agg(
             customers=("customer_id", "nunique"),
             avg_spend=("spend", "mean"),
             median_spend=("spend", "median"),
             total_spend=("spend", "sum"),
             churn_rate=("churn", "mean"),
         )
         .sort_values("total_spend", ascending=False)
)

city_summary.round(2)

In [ ]:
# transform повертає результат тієї самої довжини, що й початкові дані
city_total = clean.groupby("city")["spend"].transform("sum")
clean["share_in_city"] = clean["spend"] / city_total

clean[["customer_id", "city", "spend", "share_in_city"]].head()

## 8. merge та concat

In [ ]:
city_info = pd.DataFrame({
    "city": ["Kyiv", "Lviv", "Odesa", "Dnipro", "Kharkiv"],
    "region": ["North", "West", "South", "Center", "East"],
    "priority_market": [1, 1, 0, 0, 1],
})

merged = clean.merge(
    city_info,
    on="city",
    how="left",
    validate="many_to_one",
    indicator=True,
)

print(merged["_merge"].value_counts())
merged.head()

In [ ]:
first_half = clean.iloc[:60]
second_half = clean.iloc[60:]
restored = pd.concat([first_half, second_half], ignore_index=True)

print(clean.shape, restored.shape)

## 9. pivot_table, crosstab та melt

In [ ]:
pivot = pd.pivot_table(
    clean,
    values="spend",
    index="city",
    columns="segment",
    aggfunc="mean",
    fill_value=0,
)

pivot.round(2)

In [ ]:
churn_table = pd.crosstab(
    clean["segment"],
    clean["churn"],
    normalize="index",
)

churn_table.round(3)

In [ ]:
wide = city_summary.reset_index()[["city", "avg_spend", "churn_rate"]]
long = wide.melt(
    id_vars="city",
    var_name="metric",
    value_name="value",
)
long.head(10)

## 10. Кореляція та пошук потенційних викидів

In [ ]:
numeric_cols = ["age", "orders", "spend", "churn", "tenure_days"]
correlation = clean[numeric_cols].corr(method="pearson")
correlation.round(2)

In [ ]:
q1 = clean["spend"].quantile(0.25)
q3 = clean["spend"].quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

outliers = clean.loc[~clean["spend"].between(lower, upper) & clean["spend"].notna()]
print("Межі IQR:", round(lower, 2), round(upper, 2))
print("Кількість потенційних викидів:", len(outliers))
outliers[["customer_id", "city", "spend"]].head()

## 11. Базова візуалізація

In [ ]:
clean["spend"].plot(kind="hist", bins=20, title="Розподіл витрат")
plt.xlabel("spend")
plt.show()

In [ ]:
clean.plot.scatter(x="age", y="spend", alpha=0.6, title="Age vs spend")
plt.show()

In [ ]:
city_summary["total_spend"].sort_values().plot(
    kind="barh",
    title="Сумарні витрати за містами",
)
plt.xlabel("total spend")
plt.show()

## 12. EDA і data leakage

Нижче показано лише **правильну послідовність**: спочатку train/test split, потім статистики preprocessing оцінюються на train.

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    clean,
    test_size=0.2,
    random_state=42,
    stratify=clean["churn"],
)

train_median = train_df["age"].median()
train_age = train_df["age"].fillna(train_median)
test_age = test_df["age"].fillna(train_median)

print("train:", train_df.shape)
print("test:", test_df.shape)
print("Медіана, оцінена лише на train:", train_median)

# Завдання для студентів

Виконайте завдання нижче самостійно. Після кожного завдання додайте короткий висновок у Markdown-комірці: **що саме було виявлено і чому це важливо**.

## Завдання 1. Первинний аудит даних
1. Визначте розмір `df`.
2. Виведіть типи всіх стовпців.
3. Порахуйте кількість унікальних значень у кожному стовпці.
4. Визначте частку пропусків.
5. Назвіть щонайменше три потенційні проблеми якості даних.

In [ ]:
print(f"1. Розмір df: {df.shape[0]} рядків, {df.shape[1]} стовпців")

print("\n2. Типи стовпців:")
print(df.dtypes)

print("\n3. Кількість унікальних значень:")
print(df.nunique(dropna=False))

missing_share = df.isna().mean().sort_values(ascending=False)
print("\n4. Частка пропусків (у %):")
print((missing_share * 100).round(2).astype(str) + "%")

print("\nКількість повних дублікатів:", df.duplicated().sum())
print("Кількість дублікатів customer_id:", df.duplicated(subset=["customer_id"]).sum())

### Висновок до завдання 1
**Що було виявлено:**
- Розмір датасету становить 122 рядки та 8 ознак.
- Типи стовпців: 3 цілочисельні (`customer_id`, `orders`, `churn`), 2 з рухомою комою (`age`, `spend`), 1 часова мітка (`signup_date`), 2 текстові/об'єктні (`city`, `segment`).
- Кількість унікальних значень: `customer_id` має 120 унікальних значень при 122 рядках; стовпець `city` містить 7 варіантів значень замість очікуваних 5 міст.
- Частка пропусків: `spend` — 3.28% (4 пропуски), `age` — 2.46% (3 пропуски), `segment` — 0.82% (1 пропуск).
- **Три потенційні проблеми якості даних:**
  1. *Дублікати рядків:* присутні 2 повні дублікати (рядки з `customer_id` 1004 та 1016 продубльовано).
  2. *Неузгодженість форматування тексту:* у `city` є зайві пробіли (`' kyiv '`) та верхній регістр (`'LVIV'`), через що одне й те саме місто сприймається як різні категорії.
  3. *Неприпустимі числові значення:* рядок 88 містить від'ємне значення `spend = -150.0`, що неможливо для накопичених витрат клієнта.

**Чому це важливо:**
Первинний аудит запобігає навчанню моделей на спотворених даних. Неочищені дублікати зміщують оцінки розподілу, брудний текст штучно роздуває кількість категорій, а аномалії типу від'ємних витрат ламають статистичні метрики та бізнес-логіку.

## Завдання 2. Фільтрація
Створіть DataFrame `task2`, який містить клієнтів:
- віком від 30 до 50 років;
- із сегменту `premium` або `standard`;
- з витратами понад 600;
- лише зі стовпцями `customer_id`, `age`, `city`, `segment`, `spend`.

Відсортуйте результат за `spend` за спаданням.

In [ ]:
mask_task2 = (
    df["age"].between(30, 50)
    & df["segment"].isin(["premium", "standard"])
    & (df["spend"] > 600)
)

cols_task2 = ["customer_id", "age", "city", "segment", "spend"]
task2 = df.loc[mask_task2, cols_task2].sort_values("spend", ascending=False)

print(f"Кількість клієнтів, що відповідають умовам: {len(task2)}")
task2.head(10)

### Висновок до завдання 2
**Що було виявлено:**
- Виділено когорту високоприбуткових клієнтів середнього віку (30–50 років) преміального та стандартного сегментів.
- У відфільтрованій підмножині найвищі витрати перевищують 1000–1600 грн.

**Чому це важливо:**
Використання `.loc` у поєднанні з побітовими операторами (`&`) і методами `.between()` та `.isin()` дозволяє виконувати точну векторну фільтрацію без створення неявних копій (`SettingWithCopyWarning`). Також варто враховувати, що рядки з `NaN` автоматично відсікаються при порівняннях, тому фільтрацію до очищення даних слід робити усвідомлено.

## Завдання 3. Очищення
На копії `df` виконайте:
1. видалення повних дублікатів;
2. очищення пробілів і регістру в `city`;
3. заміну від’ємних `spend` на пропуск;
4. підрахунок кількості пропусків після очищення;
5. поясніть, які пропуски ви б видаляли, а які — імпутували.

In [ ]:
df_clean = df.copy()

df_clean = df_clean.drop_duplicates()

df_clean["city"] = df_clean["city"].str.strip().str.title()

df_clean.loc[df_clean["spend"] < 0, "spend"] = np.nan

print("Кількість пропусків після очищення:")
missing_after = df_clean.isna().sum()
print(missing_after[missing_after > 0])
print(f"\nРозмір після дедуплікації: {df_clean.shape}")

### Висновок до завдання 3
**Що було виявлено:**
- Після видалення 2 дублікатів залишилося 120 рядків.
- Кількість унікальних міст скоротилася з 7 до коректних 5 (Kyiv, Lviv, Odesa, Dnipro, Kharkiv).
- Після конвертації значення `-150.0` у `NaN` кількість пропусків у `spend` збільшилася до 5. У `age` залишилося 3 пропуски, у `segment` — 1 пропуск.

**Пояснення щодо видалення чи імпутації пропусків:**
- *Повні дублікати*: однозначно **видаляються**, оскільки це технічна помилка задвоєння записів.
- *Пропуски в `age` (3 шт., ~2.5%)*: краще **імпутувати** (наприклад, медіаною по трейну). Їх частка дуже мала, а видалення рядків призвело б до втрати валідної інформації про історію покупок клієнта.
- *Пропуски в `segment` (1 шт., <1%)*: для категоріальних ознак оптимально заповнити значенням `'unknown'` (нова категорія), щоб не приписувати випадковий статус клієнту.
- *Пропуски в `spend` (5 шт., ~4.2%)*: для EDA їх можна імпутувати медіаною або додати бінарний прапорець `spend_isna`. У ML-моделях розрахунок імпутації слід робити суворо всередині крос-валідації/пайплайну, навченого на train.

**Чому це важливо:**
Сліпе видалення рядків із пропусками призводить до зменшення розміру вибірки та ризику виникнення систематичного зсуву (selection bias).

## Завдання 4. Групова аналітика
Для кожного `segment` обчисліть:
- кількість клієнтів;
- середню кількість замовлень;
- медіанні витрати;
- сумарні витрати;
- churn rate.

Відсортуйте сегменти за churn rate від найбільшого до найменшого.

In [ ]:
segment_summary = (
    clean.groupby("segment", dropna=False)
         .agg(
             customers=("customer_id", "nunique"),
             avg_orders=("orders", "mean"),
             median_spend=("spend", "median"),
             total_spend=("spend", "sum"),
             churn_rate=("churn", "mean"),
         )
         .sort_values("churn_rate", ascending=False)
)

segment_summary.round(2)

### Висновок до завдання 4
**Що було виявлено:**
- Категорія `unknown` має лише 1 клієнта і показник churn rate = 1.0 (100%).
- Серед регулярних сегментів найбільший відтік спостерігається у `standard` (~32.6%), у той час як у `basic` (~24.1%) та `premium` (~22.2%) відтік помітно нижчий.
- Сегмент `basic` генерує найбільшу сукупну виручку (total spend) завдяки масовості (58 клієнтів), але преміум-клієнти мають вищі індивідуальні чеки.

**Чому це важливо:**
Групова агрегація висвітлює бізнес-ризики: клієнти сегмента `standard` втрачаються частіше за інших, що вказує на потребу покращення сервісу або коригування програми лояльності для середнього чека.

## Завдання 5. Ознаки на основі дат
Створіть:
- `signup_year`;
- `signup_quarter`;
- `tenure_days` станом на 07.09.2026.

Після цього визначте, у якому році зареєструвалося найбільше клієнтів.

In [ ]:
reference_date = pd.Timestamp("2026-09-07")

clean["signup_year"] = clean["signup_date"].dt.year
clean["signup_quarter"] = clean["signup_date"].dt.quarter
clean["tenure_days"] = (reference_date - clean["signup_date"]).dt.days

year_distribution = clean["signup_year"].value_counts().sort_index()
top_year = year_distribution.idxmax()
top_count = year_distribution.max()

print("Розподіл кількості реєстрацій за роками:")
print(year_distribution)
print(f"\nНайбільше клієнтів зареєструвалося у {top_year} році: {top_count} осіб.")
clean[["signup_date", "signup_year", "signup_quarter", "tenure_days"]].head()

### Висновок до завдання 5
**Що було виявлено:**
- Усі реєстрації розподілені у часовому проміжку 2022–2026 років.
- Ознака `tenure_days` відображає вік клієнта в сервісі (від ~116 до ~1698 днів).
- Розраховано рік із максимальною динамікою залучення клієнтів.

**Чому це важливо:**
Для машинного навчання календарні дати у первинному вигляді непридатні. Виділення числових компонент (рік, квартал) та перетворення на тривалість взаємодії (`tenure_days`) створює змістовні предиктори: «старі» клієнти зазвичай мають іншу поведінку відтоку, ніж новачки.

## Завдання 6. Об’єднання таблиць
Створіть окрему таблицю `segment_info` з полями:
- `segment`;
- `discount_rate`;
- `support_level`.

Об’єднайте її з очищеним набором через `merge`. Обов’язково використайте `validate` і поясніть, який тип зв’язку очікується.

In [ ]:
segment_info = pd.DataFrame({
    "segment": ["basic", "standard", "premium", "unknown"],
    "discount_rate": [0.00, 0.05, 0.15, 0.00],
    "support_level": ["standard", "priority", "dedicated", "standard"],
})

merged_task6 = clean.merge(
    segment_info,
    on="segment",
    how="left",
    validate="many_to_one",
)

print(f"Розмір після об'єднання: {merged_task6.shape}")
merged_task6[["customer_id", "segment", "discount_rate", "support_level"]].head()

### Висновок до завдання 6
**Що було виявлено:**
- До кожного клієнта успішно приєднано параметри знижки та рівня підтримки згідно з його сегментом, розмір рядків датасету зберігся (120 рядків).

**Пояснення типу зв'язку та використання validate:**
- Очікуваний тип зв'язку — **`many_to_one` (багато-до-одного)**, оскільки один і той самий сегмент зустрічається у таблиці клієнтів багато разів, але в довіднику `segment_info` кожен сегмент є унікальним первинним ключем.
- **Чому це важливо:** використання `validate='many_to_one'` страхує від критичної помилки: якщо в довіднику раптом виявиться дублікат ключа, операція `merge` виконає декартове розмноження рядків (cardinality explosion), що призведе до задвоєння фінансових показників та клієнтів. Валідація кине виняток і зупинить виконання.

## Завдання 7. Pivot table
Побудуйте `pivot_table`, де:
- рядки — `city`;
- стовпці — `segment`;
- значення — медіана `spend`.

Визначте комбінацію місто–сегмент із найбільшою медіаною витрат.

In [ ]:
pivot_task7 = pd.pivot_table(
    clean,
    values="spend",
    index="city",
    columns="segment",
    aggfunc="median",
    fill_value=0,
)

print("Зведена таблиця медіанних витрат:")
display(pivot_task7.round(2))

stacked_pivot = pivot_task7.stack()
best_city, best_seg = stacked_pivot.idxmax()
max_median = stacked_pivot.max()

print(f"\nНайбільша медіана витрат ({max_median:.2f}) у комбінації: місто {best_city}, сегмент {best_seg}")

### Висновок до завдання 7
**Що було виявлено:**
- Побудовано матрицю витрат на перетині географії та категорій клієнтів.
- Найвища медіана витрат серед регулярних категорій зафіксована у клієнтів міста **Dnipro** у сегменті **premium** (понад 970 грн).

**Чому це важливо:**
Зведені таблиці розкривають локальні взаємозв'язки, які зникають при окремому агрегуванні лише по містах або лише по сегментах. Це дає змогу виявляти цільові ринкові ніші для персоналізованого маркетингу.

## Завдання 8. Візуалізація
Побудуйте щонайменше три різні графіки:
1. розподіл `age`;
2. boxplot `spend` за сегментами;
3. churn rate за містами.

Для кожного графіка напишіть 1–2 речення інтерпретації.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

clean["age"].dropna().plot(
    kind="hist",
    bins=15,
    ax=axes[0],
    color="#4C72B0",
    edgecolor="black",
)
axes[0].set_title("Розподіл віку клієнтів (Age)")
axes[0].set_xlabel("Вік")
axes[0].set_ylabel("Кількість")

clean.boxplot(
    column="spend",
    by="segment",
    ax=axes[1],
    grid=False,
    color=dict(boxes="#4C72B0", whiskers="black", medians="#C44E52", caps="black"),
)
axes[1].set_title("Розподіл Spend за сегментами")
axes[1].set_xlabel("Сегмент")
axes[1].set_ylabel("Витрати (Spend)")
fig.suptitle("")

city_churn = clean.groupby("city")["churn"].mean().sort_values(ascending=False)
city_churn.plot(
    kind="bar",
    ax=axes[2],
    color="#DD8452",
    edgecolor="black",
)
axes[2].set_title("Рівень відтоку (Churn Rate) за містами")
axes[2].set_xlabel("Місто")
axes[2].set_ylabel("Частка відтоку (0 - 1)")
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=45)

plt.tight_layout()
plt.show()

### Висновок до завдання 8
**Інтерпретація графіків:**
1. **Гістограма `age`:** Вік клієнтів розподілений помірно рівномірно у діапазоні від 18 до 68 років з медіаною близько 44 років, без різких розривів чи однобокої концентрації.
2. **Boxplot `spend` по сегментах:** Сегмент `premium` має помітно вищий міжквартильний розмах і зсунуту вгору медіану, тоді як у `basic` та `standard` витрати концентруються нижче, проте містять поодинокі високі викиди.
3. **Стовпчикова діаграма `churn rate`:** Спостерігається відчутна регіональна різниця — в Одесі (~37.5%) та Харкові (~35.7%) відтік значно вищий, ніж у Дніпрі (~18.8%) та Львові (~21.4%).

**Чому це важливо:**
Графіки миттєво підсвічують неоднорідність підгруп та патерни відтоку, які важко помітити виключно в табличному вигляді.

## Завдання 9. Викиди
За правилом IQR знайдіть потенційні викиди `spend`.

Далі порівняйте середнє та медіану `spend` до і після **тимчасового** виключення цих рядків. Не видаляйте викиди з фінального DataFrame автоматично. Поясніть, чому.

In [ ]:
q1 = clean["spend"].quantile(0.25)
q3 = clean["spend"].quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

outliers_mask = ~clean["spend"].between(lower, upper) & clean["spend"].notna()
outliers = clean.loc[outliers_mask]

mean_before = clean["spend"].mean()
median_before = clean["spend"].median()

clean_no_outliers = clean.loc[~outliers_mask]
mean_after = clean_no_outliers["spend"].mean()
median_after = clean_no_outliers["spend"].median()

print(f"Межі IQR: нижня = {lower:.2f}, верхня = {upper:.2f}")
print(f"Кількість виявлених викидів: {len(outliers)}")
print("\nПорівняння метрик витрат (spend):")
print(f"До виключення:    Середнє = {mean_before:.2f}, Медіана = {median_before:.2f}")
print(f"Після виключення: Середнє = {mean_after:.2f}, Медіана = {median_after:.2f}")

### Висновок до завдання 9
**Що було виявлено:**
- За правилом IQR виявлено 3 потенційні викиди (значення > 1729.4 грн).
- Після їх тимчасового виключення середнє значення впало з ~672 грн до ~638 грн (зменшення на понад 30 грн), у той час як медіана практично не змінилася (~621 грн проти ~606 грн).

**Чому викиди не можна видаляти автоматично:**
1. *Природна природа даних:* ці високі витрати не є технічною помилкою сенсора чи системи, а відображають реальних найбільш платоспроможних (VIP) клієнтів магазину.
2. *Спотворення бізнес-картини:* видалення цих записів штучно занизить реальний дохід компанії та позбавить модель інформації про поведінку топових покупців.
3. *Альтернативні рішення:* замість видалення такі дані краще логарифмувати (`np.log1p`), масштабувати за допомогою `RobustScaler` або використовувати алгоритми на основі дерев рішень, які нечутливі до викидів.

## Завдання 10. Підсумковий EDA
Підготуйте короткий аналітичний висновок за набором даних. Він має містити:
- розмір і структуру даних;
- проблеми якості;
- 3–5 ключових статистичних спостережень;
- щонайменше 2 візуальні спостереження;
- потенційні ознаки для ML-моделі прогнозування `churn`;
- ризики data leakage;
- план preprocessing перед моделюванням.

In [ ]:
print("Баланс цільової змінної churn:")
print(clean["churn"].value_counts(normalize=True).round(3))

print("\nКореляція ознак із цільовою змінною churn:")
num_cols = ["age", "orders", "spend", "tenure_days", "spend_per_order", "churn"]
corr = clean[num_cols].corr()["churn"].sort_values(ascending=False)
print(corr.round(3))

### Підсумковий аналітичний звіт (EDA)

1. **Розмір і структура даних:**
   - Початковий датасет містив 122 рядки та 8 колонок. Після видалення дублікатів та створення нових ознак фінальний набір містить 120 спостережень та 13 ознак (демографічні, поведінкові, транзакційні та цільова мітка `churn`).

2. **Проблеми якості даних:**
   - Наявність 2 повних дублікатів рядків (успішно видалені).
   - Неакуратний текст у стовпці `city` (зайві пробіли та різний регістр призводили до 7 міст замість 5).
   - Неприпустиме від'ємне значення `spend = -150.0` (переведено в `NaN`).
   - Пропуски в `spend` (5 шт.), `age` (3 шт.) та `segment` (1 шт.).

3. **Ключові статистичні спостереження:**
   - Цільова змінна незбалансована: частка відтоку становить ~26.3% (базовий клас 0 — 73.7%).
   - Середні витрати (~672 грн) помітно перевищують медіану (~621 грн) через наявність важкого правого хвоста розподілу.
   - Сегмент `standard` показує найвищий рівень відтоку серед основних клієнтів (32.6%).
   - Лінійні кореляції між віком, витратами та замовленнями близькі до нуля, зв'язки з відтоком є нелінійними.

4. **Візуальні спостереження:**
   - Яскраво виражена географічна неоднорідність: Одеса та Харків мають відтік понад 35%, тоді як Дніпро — менше 19%.
   - Розподіл витрат сильно асиметричний, сегмент `premium` характеризується найбільшою дисперсією.

5. **Потенційні ознаки для ML-моделі прогнозування `churn`:**
   - `tenure_days` (найбільша від'ємна кореляція з відтоком: $r = -0.15$ — лояльні клієнти йдуть рідше);
   - `spend_per_order` (середній чек);
   - `city` (важливий фактор ризику з огляду на різницю за регіонами);
   - `segment` та `orders`.

6. **Ризики Data Leakage (витоку даних):**
   - Обчислення медіани `age` або `spend` по всьому датасету до розбиття вибірки призводить до просочування інформації з test у train.
   - Створення ознак на основі агрегацій (`share_in_city`) на повному датасеті використовує суми по місту з урахуванням тестових клієнтів.

7. **План Preprocessing перед моделюванням:**
   - *Крок 1:* Первинне очищення сирих даних (дедуплікація, очищення тексту, валідація від'ємних значень).
   - *Крок 2:* Стратифікований поділ на `train` і `test` (`stratify=churn`).
   - *Крок 3:* Побудова пайплайну передобробки (`ColumnTransformer`):
     - Числові ознаки: `SimpleImputer(strategy='median')` -> `RobustScaler()` (для стійкості до викидів);
     - Категоріальні ознаки: `SimpleImputer(strategy='constant', fill_value='unknown')` -> `OneHotEncoder(handle_unknown='ignore')`.
   - *Крок 4:* Валідація моделей виключно через Stratified K-Fold на train-вибірці.

## Критерії самоперевірки

Робота виконана якісно, якщо:
- код запускається послідовно без помилок;
- вихідний DataFrame не змінюється випадково через неявні присвоєння;
- для вибірки за умовами використано `loc`;
- пропуски й дублікати не обробляються без пояснення;
- `merge` перевіряє очікувану кардинальність через `validate`;
- графіки мають назви та зрозумілі осі;
- висновки містять інтерпретацію, а не лише перелік чисел;
- при переході до ML враховано ризик витоку даних.